# Pipeline A: Incident Escalation Risk Predictor

**Object Name:** Incident Escalation / Forward-Looking Severity Predictor

**Goal:** Identify residents whose next incident report is likely to be of **High or Critical** severity based on their recent behavioral, environmental, and health history.

**Business Problem:** Casceworkers are overwhelmed with low-level incident reports. This pipeline acts as a weekly alert system to prioritize interventions for residents at risk of imminent escalation (the "falling through the cracks" scenario).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

# 1. Setup Data Paths
REPO_ROOT = Path.cwd().parent.parent
DATA_DIR = REPO_ROOT / "data" / "lighthouse_csv_v7"

def load_table(name):
    fp = DATA_DIR / f"{name}.csv"
    return pd.read_csv(fp) if fp.exists() else pd.DataFrame()

incidents = load_table("incident_reports")
residents = load_table("residents")
recordings = load_table("process_recordings")
visitations = load_table("home_visitations")
health = load_table("health_wellbeing_records")

## Feature Engineering

We build features at the resident-incident level. For each incident, we look at the resident's *prior* history to ensure no temporal leakage.

In [ ]:
# Convert dates
incidents['incident_date'] = pd.to_datetime(incidents['incident_date'])
recordings['recording_date'] = pd.to_datetime(recordings['recording_date'])
visitations['visit_date'] = pd.to_datetime(visitations['visit_date'])
health['assessment_date'] = pd.to_datetime(health['assessment_date'])

# Target: Severity is High or Critical
incidents['is_serious'] = incidents['severity'].apply(lambda x: 1 if str(x).lower() in ['high', 'critical'] else 0)

def get_resident_features_at_time(resident_id, reference_date):
    # Frequency in last 30, 60, 90 days
    hist = incidents[(incidents['resident_id'] == resident_id) & (incidents['incident_date'] < reference_date)]
    f30 = len(hist[hist['incident_date'] >= reference_date - pd.Timedelta(days=30)])
    f60 = len(hist[hist['incident_date'] >= reference_date - pd.Timedelta(days=60)])
    
    # Recording concerns
    rec = recordings[(recordings['resident_id'] == resident_id) & (recordings['recording_date'] < reference_date)]
    concerns_rate = rec['concerns_flagged'].mean() if not rec.empty else 0
    
    # Latest health score
    h_rec = health[(health['resident_id'] == resident_id) & (health['assessment_date'] < reference_date)].sort_values('assessment_date').tail(1)
    h_score = h_rec['general_health_score'].values[0] if not h_rec.empty else np.nan
    
    return pd.Series({
        'inc_count_30d': f30,
        'inc_count_60d': f60,
        'concerns_flagged_rate': concerns_rate,
        'latest_health_score': h_score
    })

# Build the dataset (Sampled for speed in notebook)
data = incidents.copy()
features = data.apply(lambda row: get_resident_features_at_time(row['resident_id'], row['incident_date']), axis=1)
df = pd.concat([data, features], axis=1).dropna(subset=['is_serious'])

## Modeling

1. **Explanatory:** Logistic Regression with L2 regularization to find the most significant predictors of escalation.
2. **Predictive:** HistGradientBoostingClassifier for robust weekly scoring.

In [ ]:
X = df[['inc_count_30d', 'inc_count_60d', 'concerns_flagged_rate', 'latest_health_score']]
y = df['is_serious']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Pipeline
numeric_features = X.columns.tolist()
preprocessor = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_features)
])

lr_model = Pipeline([('prep', preprocessor), ('clf', LogisticRegression(class_weight='balanced'))])
gb_model = Pipeline([('prep', preprocessor), ('clf', HistGradientBoostingClassifier(class_weight='balanced'))])

lr_model.fit(X_train, y_train)
gb_model.fit(X_train, y_train)

print("Logistic Regression AUC:", roc_auc_score(y_test, lr_model.predict_proba(X_test)[:, 1]))
print("Gradient Boosting AUC:", roc_auc_score(y_test, gb_model.predict_proba(X_test)[:, 1]))